# Fast Comparison (Tuned Backbone): Linear vs KAN

Mejoras aplicadas:
- Embeddings desde `resnet18_tuned_best.pt` (no ImageNet puro)
- Estandarizaci?n z-score de embeddings
- Umbral calibrado por recall objetivo en validaci?n
- Comparaci?n justa en test con mismo criterio cl?nico


In [1]:
# Si falta KAN:
# %pip install pykan


In [2]:
from pathlib import Path
from collections import Counter
import importlib.util
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, f1_score, precision_score, confusion_matrix, classification_report, precision_recall_curve

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
KAN_AVAILABLE = importlib.util.find_spec('kan') is not None
print('KAN available:', KAN_AVAILABLE)
DEVICE_BACKBONE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_HEAD = torch.device('cpu')
print('Backbone device:', DEVICE_BACKBONE, '| Head device:', DEVICE_HEAD)


KAN available: True
Backbone device: cuda | Head device: cpu


In [3]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
TRAIN_CSV = ROOT / 'src/data/processed/manifest_train.csv'
TEST_CSV = ROOT / 'src/data/processed/manifest_test.csv'
TUNED_BACKBONE = ROOT / 'reports/models/resnet18_tuned_best.pt'

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
patients = train_df['patient_id'].dropna().unique()
train_pat, val_pat = train_test_split(patients, test_size=0.2, random_state=SEED)
tr_df = train_df[train_df['patient_id'].isin(train_pat)].copy()
val_df = train_df[train_df['patient_id'].isin(val_pat)].copy()
print('Train:', tr_df.shape, tr_df['label_name'].value_counts().to_dict())
print('Val:', val_df.shape, val_df['label_name'].value_counts().to_dict())
print('Test:', test_df.shape, test_df['label_name'].value_counts().to_dict())
print('Tuned model exists:', TUNED_BACKBONE.exists())


Train: (2315, 10) {'BENIGN': 1383, 'MALIGNANT': 932}
Val: (549, 10) {'BENIGN': 300, 'MALIGNANT': 249}
Test: (422, 10) {'BENIGN': 248, 'MALIGNANT': 174}
Tuned model exists: True


In [4]:
class MammographyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        x = Image.open(r['image_path_local']).convert('RGB')
        x = self.transform(x)
        y = int(r['label'])
        return x, y

tfm = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_img_loader = DataLoader(MammographyDataset(tr_df, tfm), batch_size=32, shuffle=False, num_workers=0)
val_img_loader = DataLoader(MammographyDataset(val_df, tfm), batch_size=32, shuffle=False, num_workers=0)
test_img_loader = DataLoader(MammographyDataset(test_df, tfm), batch_size=32, shuffle=False, num_workers=0)


In [5]:
# Cargar modelo tuned y usar solo extractor de features
full = models.resnet18(weights=None)
full.fc = nn.Linear(full.fc.in_features, 2)
full.load_state_dict(torch.load(TUNED_BACKBONE, map_location=DEVICE_BACKBONE))
extractor = nn.Sequential(*list(full.children())[:-1]).to(DEVICE_BACKBONE).eval()
for p in extractor.parameters():
    p.requires_grad = False

def extract_embeddings(loader):
    Xs, ys = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE_BACKBONE)
            z = extractor(x).flatten(1)
            Xs.append(z.cpu())
            ys.append(y.cpu())
    return torch.cat(Xs,0), torch.cat(ys,0)

Xtr, ytr = extract_embeddings(train_img_loader)
Xva, yva = extract_embeddings(val_img_loader)
Xte, yte = extract_embeddings(test_img_loader)
print(Xtr.shape, Xva.shape, Xte.shape)


torch.Size([2315, 512]) torch.Size([549, 512]) torch.Size([422, 512])


In [6]:
# Estandarizaci?n con stats de train
mu = Xtr.mean(dim=0, keepdim=True)
sigma = Xtr.std(dim=0, keepdim=True).clamp_min(1e-6)
Xtr = (Xtr - mu) / sigma
Xva = (Xva - mu) / sigma
Xte = (Xte - mu) / sigma

Xtr_h, ytr_h = Xtr.to(DEVICE_HEAD), ytr.to(DEVICE_HEAD)
Xva_h, yva_h = Xva.to(DEVICE_HEAD), yva.to(DEVICE_HEAD)
Xte_h, yte_h = Xte.to(DEVICE_HEAD), yte.to(DEVICE_HEAD)

c = Counter(ytr.tolist())
w = torch.tensor([len(ytr)/(2*c[0]), len(ytr)/(2*c[1])], dtype=torch.float32, device=DEVICE_HEAD)
criterion = nn.CrossEntropyLoss(weight=w)
w


tensor([0.8369, 1.2420])

In [7]:
def metrics_from_logits(y_true_t, logits_t, thr=0.5):
    y_true = y_true_t.cpu().numpy()
    prob = torch.softmax(logits_t, dim=1)[:,1].detach().cpu().numpy()
    pred = (prob >= thr).astype(int)
    return {
        'acc': accuracy_score(y_true, pred),
        'auc': roc_auc_score(y_true, prob),
        'recall_malignant': recall_score(y_true, pred, pos_label=1),
        'precision_malignant': precision_score(y_true, pred, pos_label=1),
        'f1_malignant': f1_score(y_true, pred, pos_label=1),
        'y_true': y_true, 'y_pred': pred, 'y_prob': prob
    }

def choose_threshold_by_recall(y_true, y_prob, target_recall=0.80):
    p, r, t = precision_recall_curve(y_true, y_prob)
    r_t = r[:-1]; p_t = p[:-1]
    idx = np.where(r_t >= target_recall)[0]
    if len(idx) == 0:
        bi = int(np.argmax(r_t))
    else:
        bi = int(idx[np.argmax(p_t[idx])])
    return float(t[bi])

def print_summary(name, m):
    print(f"{name}: acc={m['acc']:.4f} auc={m['auc']:.4f} rec_mal={m['recall_malignant']:.4f} prec_mal={m['precision_malignant']:.4f} f1_mal={m['f1_malignant']:.4f}")


In [8]:
# Linear head
lin = nn.Linear(512,2).to(DEVICE_HEAD)
opt = torch.optim.Adam(lin.parameters(), lr=1e-3)
sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)
best_auc, best_state, wait, patience = -1.0, None, 0, 4
for ep in range(1,21):
    lin.train(); opt.zero_grad()
    lg = lin(Xtr_h); loss = criterion(lg, ytr_h); loss.backward(); opt.step()
    lin.eval();
    with torch.no_grad(): va_logits = lin(Xva_h)
    vm = metrics_from_logits(yva_h, va_logits, thr=0.5)
    sch.step(vm['auc'])
    print(f"Linear E{ep:02d} loss={loss.item():.4f} va_auc={vm['auc']:.4f}")
    if vm['auc'] > best_auc:
        best_auc, wait = vm['auc'], 0
        best_state = {k:v.cpu().clone() for k,v in lin.state_dict().items()}
    else:
        wait += 1
        if wait >= patience:
            print('Early stop linear'); break
lin.load_state_dict(best_state)
lin.eval()
with torch.no_grad():
    va_logits_lin = lin(Xva_h)
    te_logits_lin = lin(Xte_h)
mva_lin = metrics_from_logits(yva_h, va_logits_lin, thr=0.5)
thr_lin = choose_threshold_by_recall(mva_lin['y_true'], mva_lin['y_prob'], target_recall=0.80)
mte_lin = metrics_from_logits(yte_h, te_logits_lin, thr=thr_lin)
print('Linear threshold:', round(thr_lin,4))
print_summary('Linear test', mte_lin)


Linear E01 loss=0.8224 va_auc=0.5208
Linear E02 loss=0.7404 va_auc=0.5706
Linear E03 loss=0.6682 va_auc=0.6154
Linear E04 loss=0.6057 va_auc=0.6486
Linear E05 loss=0.5522 va_auc=0.6712
Linear E06 loss=0.5069 va_auc=0.6844
Linear E07 loss=0.4689 va_auc=0.6936
Linear E08 loss=0.4371 va_auc=0.7003
Linear E09 loss=0.4106 va_auc=0.7055
Linear E10 loss=0.3885 va_auc=0.7074
Linear E11 loss=0.3700 va_auc=0.7104
Linear E12 loss=0.3544 va_auc=0.7119
Linear E13 loss=0.3412 va_auc=0.7131
Linear E14 loss=0.3299 va_auc=0.7138
Linear E15 loss=0.3202 va_auc=0.7141
Linear E16 loss=0.3119 va_auc=0.7142
Linear E17 loss=0.3046 va_auc=0.7151
Linear E18 loss=0.2982 va_auc=0.7152
Linear E19 loss=0.2926 va_auc=0.7157
Linear E20 loss=0.2875 va_auc=0.7163
Linear threshold: 0.3576
Linear test: acc=0.6043 auc=0.7022 rec_mal=0.8506 prec_mal=0.5121 f1_mal=0.6393


In [9]:
if not KAN_AVAILABLE:
    raise RuntimeError('KAN no instalado. Ejecuta `%pip install pykan`, reinicia kernel y corre de nuevo.')
from kan import KAN
kan = KAN(width=[512,16,2], grid=3, k=3).to(DEVICE_HEAD)
optk = torch.optim.Adam(kan.parameters(), lr=1e-3)
schk = torch.optim.lr_scheduler.ReduceLROnPlateau(optk, mode='max', factor=0.5, patience=2)
best_auc, best_state, wait, patience = -1.0, None, 0, 4
for ep in range(1,16):
    kan.train(); optk.zero_grad()
    lg = kan(Xtr_h); loss = criterion(lg, ytr_h); loss.backward(); optk.step()
    kan.eval();
    with torch.no_grad(): va_logits = kan(Xva_h)
    vm = metrics_from_logits(yva_h, va_logits, thr=0.5)
    schk.step(vm['auc'])
    print(f"KAN E{ep:02d} loss={loss.item():.4f} va_auc={vm['auc']:.4f}")
    if vm['auc'] > best_auc:
        best_auc, wait = vm['auc'], 0
        best_state = {k:v.cpu().clone() for k,v in kan.state_dict().items()}
    else:
        wait += 1
        if wait >= patience:
            print('Early stop KAN'); break
kan.load_state_dict(best_state)
kan.eval()
with torch.no_grad():
    va_logits_kan = kan(Xva_h)
    te_logits_kan = kan(Xte_h)
mva_kan = metrics_from_logits(yva_h, va_logits_kan, thr=0.5)
thr_kan = choose_threshold_by_recall(mva_kan['y_true'], mva_kan['y_prob'], target_recall=0.80)
mte_kan = metrics_from_logits(yte_h, te_logits_kan, thr=thr_kan)
print('KAN threshold:', round(thr_kan,4))
print_summary('KAN test', mte_kan)


checkpoint directory created: ./model
saving model version 0.0
KAN E01 loss=0.6827 va_auc=0.5759
KAN E02 loss=0.6510 va_auc=0.6323
KAN E03 loss=0.6212 va_auc=0.6613
KAN E04 loss=0.5924 va_auc=0.6786
KAN E05 loss=0.5643 va_auc=0.6895
KAN E06 loss=0.5368 va_auc=0.6965
KAN E07 loss=0.5099 va_auc=0.7005
KAN E08 loss=0.4839 va_auc=0.7040
KAN E09 loss=0.4589 va_auc=0.7062
KAN E10 loss=0.4353 va_auc=0.7088
KAN E11 loss=0.4131 va_auc=0.7105
KAN E12 loss=0.3926 va_auc=0.7121
KAN E13 loss=0.3737 va_auc=0.7131
KAN E14 loss=0.3566 va_auc=0.7135
KAN E15 loss=0.3410 va_auc=0.7141
KAN threshold: 0.3843
KAN test: acc=0.6090 auc=0.6949 rec_mal=0.8448 prec_mal=0.5158 f1_mal=0.6405


In [10]:
cmp = pd.DataFrame([
    {'model':'Linear (tuned feats)', 'thr':thr_lin, **{k:mte_lin[k] for k in ['acc','auc','recall_malignant','precision_malignant','f1_malignant']}},
    {'model':'KAN (tuned feats)', 'thr':thr_kan, **{k:mte_kan[k] for k in ['acc','auc','recall_malignant','precision_malignant','f1_malignant']}},
])
cmp


,model,thr,acc,auc,recall_malignant,precision_malignant,f1_malignant
0,Linear (tuned feats),0.357619,0.604265,0.702215,0.850575,0.512111,0.639309
1,KAN (tuned feats),0.384311,0.609005,0.694869,0.844828,0.515789,0.640523


In [11]:
print('Linear confusion matrix')
print(confusion_matrix(mte_lin['y_true'], mte_lin['y_pred']))
print(classification_report(mte_lin['y_true'], mte_lin['y_pred'], target_names=['BENIGN','MALIGNANT']))
print('KAN confusion matrix')
print(confusion_matrix(mte_kan['y_true'], mte_kan['y_pred']))
print(classification_report(mte_kan['y_true'], mte_kan['y_pred'], target_names=['BENIGN','MALIGNANT']))


Linear confusion matrix
[[107 141]
 [ 26 148]]
              precision    recall  f1-score   support

      BENIGN       0.80      0.43      0.56       248
   MALIGNANT       0.51      0.85      0.64       174

    accuracy                           0.60       422
   macro avg       0.66      0.64      0.60       422
weighted avg       0.68      0.60      0.59       422

KAN confusion matrix
[[110 138]
 [ 27 147]]
              precision    recall  f1-score   support

      BENIGN       0.80      0.44      0.57       248
   MALIGNANT       0.52      0.84      0.64       174

    accuracy                           0.61       422
   macro avg       0.66      0.64      0.61       422
weighted avg       0.68      0.61      0.60       422

